# 01B — Vérité terrain spatiale indépendante (tâches 09 et 10)


Ce notebook ne charge aucune prédiction de modèle. Les masques indépendants sont créés par le script d’annotation ; ce notebook les valide, mesure l’accord, exige l’adjudication et les verrouille.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd

from src import experiment_config as cfg
from src.data.database import build_object_summary
from src.decision.truth import (
    build_annotation_agreement_table,
    build_spatial_ground_truth_lock,
    build_spatial_ground_truth_manifest,
    extract_reference_components,
    select_annotation_subset,
    select_double_annotation_images,
    validate_annotation_adjudication,
    verify_spatial_ground_truth_lock,
)
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import (
    build_protocol_configuration,
    sha256_file,
    sha256_payload,
)

QC_DIR = PROJECT_ROOT.joinpath(*cfg.QC_RESULTS_RELATIVE_DIR)
RESULTS_DIR = PROJECT_ROOT.joinpath(*cfg.SPATIAL_GT_RESULTS_RELATIVE_DIR)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT = {
    key: RESULTS_DIR / filename
    for key, filename in cfg.SPATIAL_GT_OUTPUT_FILENAMES.items()
}
ANNOTATION_DIR = PROJECT_ROOT.joinpath(*cfg.SPATIAL_GT_ANNOTATION_RELATIVE_DIR)
ROI_DIR = ANNOTATION_DIR / "roi_masks"
TARGET_MASK_DIR = ANNOTATION_DIR / "target_masks"
VALIDITY_MASK_DIR = ANNOTATION_DIR / "validity_masks"
METADATA_DIR = ANNOTATION_DIR / "metadata"
ANNOTATION_PROTOCOL_PATH = PROJECT_ROOT.joinpath(
    *cfg.SPATIAL_GT_ANNOTATION_PROTOCOL_RELATIVE_PATH
)
annotation_protocol = json.loads(ANNOTATION_PROTOCOL_PATH.read_text("utf-8"))
annotation_protocol_sha256 = sha256_file(ANNOTATION_PROTOCOL_PATH)
split_manifest = pd.read_parquet(
    QC_DIR / cfg.QC_OUTPUT_FILENAMES["split_manifest"]
)
object_db, image_db = load_nir_uco_h5(
    PROJECT_ROOT.joinpath(*cfg.DATABASE_H5_RELATIVE_PATH),
    reconstruct_heavy_object_arrays=False,
)
subset = select_annotation_subset(
    split_manifest,
    build_object_summary(object_db),
)


In [2]:
records = []
double_images = select_double_annotation_images(subset)
for source_image in subset["source_image"].astype(str):
    annotators = ("annotator_1", "annotator_2") if source_image in double_images else ("annotator_1",)
    for annotator in annotators:
        reference_id = f"{source_image}__{annotator}"
        roi_path = ROI_DIR / f"{source_image}.npy"
        target_mask_path = TARGET_MASK_DIR / f"{reference_id}.npy"
        validity_mask_path = VALIDITY_MASK_DIR / f"{reference_id}.npy"
        metadata_path = METADATA_DIR / f"{reference_id}.json"
        required_paths = (
            roi_path, target_mask_path, validity_mask_path, metadata_path,
        )
        records.append(
            {
                "reference_id": reference_id,
                "source_image": source_image,
                "source_class": str(image_db[source_image]["nut_type"]),
                "annotator_id": annotator,
                "truth_level": "pixel_annotated",
                "target_class": cfg.SPATIAL_GT_TARGET_CLASS,
                "annotated_class": cfg.SPATIAL_GT_ANNOTATED_CLASS,
                "positive_value": cfg.SPATIAL_GT_POSITIVE_VALUE,
                "positive_class": cfg.SPATIAL_GT_POSITIVE_CLASS,
                "negative_value": cfg.SPATIAL_GT_NEGATIVE_VALUE,
                "roi_mask": roi_path,
                "target_mask": target_mask_path,
                "validity_mask": validity_mask_path,
                "metadata": metadata_path,
                "annotation_protocol_sha256": annotation_protocol_sha256,
                "image_shape": image_db[source_image]["labels"].shape,
                "object_area": image_db[source_image]["labels"] > 0,
                "status": "accepted" if all(path.exists() for path in required_paths) else "pending",
            }
        )
annotation_manifest = build_spatial_ground_truth_manifest(records)
component_tables = []
for record in records:
    if record["status"] != "accepted":
        continue
    component_tables.append(
        extract_reference_components(
            np.load(record["target_mask"], allow_pickle=False),
            reference_id=record["reference_id"],
        )
    )
components = (
    pd.concat(component_tables, ignore_index=True)
    if component_tables
    else pd.DataFrame(
        columns=cfg.SPATIAL_GT_COMPONENT_COLUMNS
    )
)
agreement = build_annotation_agreement_table(annotation_manifest)
adjudication = (
    pd.read_parquet(OUTPUT["adjudication"])
    if OUTPUT["adjudication"].exists()
    else pd.DataFrame(columns=cfg.SPATIAL_GT_ADJUDICATION_COLUMNS)
)
annotation_manifest.to_parquet(OUTPUT["manifest"], index=False)
components.to_parquet(OUTPUT["components"], index=False)
agreement.to_parquet(OUTPUT["agreement"], index=False)
adjudication.to_parquet(OUTPUT["adjudication"], index=False)
OUTPUT["annotation_protocol"].write_text(
    json.dumps(annotation_protocol, ensure_ascii=False, indent=2, sort_keys=True),
    encoding="utf-8",
)


2165

In [3]:
if annotation_manifest["status"].eq("pending").any():
    pending = annotation_manifest.loc[
        annotation_manifest["status"].eq("pending"),
        [
            "reference_id", "target_mask_path",
            "validity_mask_path", "metadata_path",
        ],
    ]
    raise RuntimeError(
        "Annotations pending: create the independent .npy masks listed in "
        f"{OUTPUT['manifest']}. Pending={pending.to_dict('records')}"
    )
expected_double = {
    source for source in double_images
    if source in set(annotation_manifest["source_image"].astype(str))
}
observed_double = set(agreement.get("source_image", pd.Series(dtype=str)).astype(str))
if expected_double != observed_double:
    raise RuntimeError("Agreement has not been computed for every double annotation.")
double_target_images = {
    source for source in double_images
    if str(image_db[source].get("nut_type")) == cfg.SPATIAL_GT_TARGET_CLASS
}
if not double_target_images:
    raise RuntimeError(
        "Agreement requires at least one double-annotated peanut image."
    )
validate_annotation_adjudication(agreement, adjudication)
configuration_hash = sha256_payload(build_protocol_configuration())
lock = build_spatial_ground_truth_lock(
    annotation_manifest,
    components,
    agreement,
    adjudication,
    configuration_hash=configuration_hash,
)
OUTPUT["lock"].write_text(
    json.dumps(lock, ensure_ascii=False, indent=2, sort_keys=True),
    encoding="utf-8",
)
verify_spatial_ground_truth_lock(
    OUTPUT["lock"],
    annotation_manifest,
    components,
    agreement,
    adjudication,
)
lock


{'protocol_version': '8tracks_v1',
 'annotation_protocol_version': 'spatial_gt_v1',
 'configuration_hash': '0b6274652e9740ab6a4308a73c0f4214541aa5a446db3053638c718ebcd9de05',
 'annotation_file_hashes': {'almond4__annotator_1': {'roi_mask': 'fad049a7d36849e4126283fd6748f779079cb361e0bec3b54884a11178a63dd7',
   'target_mask': '1daacb0a9212d79945e73808869938140e7b4769f90ce16d45840b032fe98e9f',
   'validity_mask': 'fad049a7d36849e4126283fd6748f779079cb361e0bec3b54884a11178a63dd7',
   'metadata': 'a86bd6e2c986196d851d07188509049bbd0247b76927ab7b8772f9093590abeb'},
  'almond4__annotator_2': {'roi_mask': 'fad049a7d36849e4126283fd6748f779079cb361e0bec3b54884a11178a63dd7',
   'target_mask': '1daacb0a9212d79945e73808869938140e7b4769f90ce16d45840b032fe98e9f',
   'validity_mask': 'fad049a7d36849e4126283fd6748f779079cb361e0bec3b54884a11178a63dd7',
   'metadata': 'ea6a69739c7a3876428a721c825af04d60092aff8c4957695d124ce3990e0e80'},
  'peanut4__annotator_1': {'roi_mask': '30f59b9434da615f4f0716a9b1f70